In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
import polars as pl
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_rows', None)

In [4]:
import polars as pl
import os
import glob
import time

folder_path = r"E:\WE Work\Abdalrhman\ML\New folder\data_Q3_2025"
output_file = "seagate_full_data.csv"

all_files = glob.glob(os.path.join(folder_path, "*.csv"))
total_files = len(all_files)

print(f"بدء المعالجة لعدد {total_files} ملف بطريقة حفظ الذاكرة (Memory-Safe)...")

start_time = time.time()
total_rows = 0
is_first_write = True # متغير عشان يكتب أسماء العواميد في أول مرة بس

# خطوة أمان: حذف الملف القديم لو موجود عشان منكتبش الداتا عليه مرتين بالغلط
if os.path.exists(output_file):
    os.remove(output_file)

# ==========================================
# 2. حلقة المعالجة والحفظ المباشر
# ==========================================
for i, file_path in enumerate(all_files, start=1):
    file_name = os.path.basename(file_path)
    
    try:
        # قراءة وفلترة الملف الحالي
        df_filtered = (
            pl.scan_csv(file_path, infer_schema_length=0)
            .filter(pl.col("model").str.starts_with("ST"))
            .collect()
        )
        
        current_rows = df_filtered.height
        
        if current_rows > 0:
            # السر كله هنا: فتح الملف في وضع الإضافة (append)
            # بيكتب الداتا على الهارد مباشرة ويفرغ الرامات
            with open(output_file, mode="ab") as f:
                df_filtered.write_csv(f, include_header=is_first_write)
            
            is_first_write = False # عشان ميكتبش أسماء العواميد تاني في الملفات الجاية
            total_rows += current_rows
            
        print(f"[{i:02d}/{total_files}] تمت معالجة: {file_name} | استخراج: {current_rows} صف")
        
    except Exception as e:
        print(f"[{i:02d}/{total_files}] حدث خطأ في ملف {file_name}: {e}")

# ==========================================
# 3. التقرير النهائي
# ==========================================
end_time = time.time()
print("-" * 60)
print("✅ تمت العملية بنجاح وبدون استهلاك للرامات!")
print(f"⏱️ الوقت المستغرق: {round(end_time - start_time, 2)} ثانية.")
print(f"💾 الملف النهائي جاهز: {output_file}")
print(f"📊 إجمالي عدد الصفوف النهائي: {total_rows} صف")
print("-" * 60)

بدء المعالجة لعدد 92 ملف بطريقة حفظ الذاكرة (Memory-Safe)...
[01/92] تمت معالجة: 2025-07-01.csv | استخراج: 110782 صف
[02/92] تمت معالجة: 2025-07-02.csv | استخراج: 110835 صف
[03/92] تمت معالجة: 2025-07-03.csv | استخراج: 110781 صف
[04/92] تمت معالجة: 2025-07-04.csv | استخراج: 110804 صف
[05/92] تمت معالجة: 2025-07-05.csv | استخراج: 110749 صف
[06/92] تمت معالجة: 2025-07-06.csv | استخراج: 110763 صف
[07/92] تمت معالجة: 2025-07-07.csv | استخراج: 110765 صف
[08/92] تمت معالجة: 2025-07-08.csv | استخراج: 110728 صف
[09/92] تمت معالجة: 2025-07-09.csv | استخراج: 110776 صف
[10/92] تمت معالجة: 2025-07-10.csv | استخراج: 110810 صف
[11/92] تمت معالجة: 2025-07-11.csv | استخراج: 110815 صف
[12/92] تمت معالجة: 2025-07-12.csv | استخراج: 110749 صف
[13/92] تمت معالجة: 2025-07-13.csv | استخراج: 110791 صف
[14/92] تمت معالجة: 2025-07-14.csv | استخراج: 110702 صف
[15/92] تمت معالجة: 2025-07-15.csv | استخراج: 110698 صف
[16/92] تمت معالجة: 2025-07-16.csv | استخراج: 110742 صف
[17/92] تمت معالجة: 2025-07-17.csv | استخرا

In [8]:
file_path = "seagate_full_data.csv"
output_clean = "seagate_cleaned_cols3.csv"

print("جاري فحص الأعمدة وحساب نسبة الـ Nulls...")

df_lazy = pl.scan_csv(file_path)

df_sample = df_lazy.head(10000000).collect()

total_rows = df_sample.height
null_percentages = (df_sample.null_count() / total_rows * 100).to_dicts()[0]

threshold = 80.0
columns_to_keep = [col for col, null_pct in null_percentages.items() if null_pct < threshold]

print(f"إجمالي الأعمدة الأصلي: {len(df_sample.columns)}")
print(f"الأعمدة المتبقية بعد الفلترة: {len(columns_to_keep)}")
print(f"تم التخلص من {len(df_sample.columns) - len(columns_to_keep)} عمود!")

# 5. حفظ البيانات النظيفة بالعمواميد المفيدة فقط
df_lazy.select(columns_to_keep).collect(streaming=True).write_csv(output_clean)

print(f"تم حفظ الملف النظيف: {output_clean}")

جاري فحص الأعمدة وحساب نسبة الـ Nulls...
إجمالي الأعمدة الأصلي: 197
الأعمدة المتبقية بعد الفلترة: 63
تم التخلص من 134 عمود!


C:\Users\INTEGRA\AppData\Local\Temp\ipykernel_14164\3616534477.py:21: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_lazy.select(columns_to_keep).collect(streaming=True).write_csv(output_clean)


تم حفظ الملف النظيف: seagate_cleaned_cols3.csv
